# Visualizing the Titanic Dataset

With the basic EDA questions answered (shape, missing values, correlations — see the previous notebook), this one picks specific columns and visualizes them, since a plot often reveals a distribution's shape faster than any table of numbers can.

## Setup

In [1]:
import pandas as pd
import seaborn as sns

In [2]:
df = pd.read_csv('train.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

A quick look at the raw data before plotting anything.

In [ ]:
df.head()

# 1. Categorical Data

**Categorical** columns hold a fixed set of labels (e.g. `Embarked` has only `S`, `C`, `Q`) rather than a continuous range of numbers. The two go-to plots for a single categorical column are a **countplot** (bar of raw counts) and a **pie chart** (proportions).

## a. Countplot

`sns.countplot()` shows how many rows fall into each category — a quick way to spot class imbalance (here, far more passengers embarked at `S` (Southampton) than `C` or `Q`).

### ⚠️ Note: Passing a Series Positionally Is Deprecated in Modern Seaborn

`sns.countplot(df['Embarked'])` passes the column as a positional argument. Since **seaborn 0.12**, plotting functions expect data via explicit keyword arguments — the positional form now raises a `TypeError` on current seaborn. Use `x=` (and pass the whole `data=df` when convenient) instead:

In [ ]:
sns.countplot(df['Embarked'])
#df['Survived'].value_counts().plot(kind='bar')

In [ ]:
sns.countplot(x='Embarked', data=df)


## b. PieChart

A pie chart works directly off `.value_counts()` output using pandas' own `.plot(kind='pie')` — `autopct='%.2f'` labels each slice with its percentage.

In [ ]:
df['Sex'].value_counts().plot(kind='pie',autopct='%.2f')

<div style="border-top:black 2px solid"></div>

# 2. Numerical Data

**Numerical** columns hold continuous values (e.g. `Age`, `Fare`). The typical starting plots are a **histogram** (distribution shape via binned counts), a **distribution/density plot**, and a **boxplot** (spread and outliers).

## a. Histogram

A histogram buckets values into ranges (`bins=5` here means 5 buckets across `Age`'s range) and counts how many rows fall in each — the classic way to see whether a numeric column is roughly symmetric, skewed, or multi-modal.

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df['Age'],bins=5)

## b. Distplot

### ⚠️ Important: `sns.distplot()` Has Been Removed

`sns.distplot()` was deprecated starting in seaborn 0.11 and has since been **removed entirely** — running `sns.distplot(df['Age'])` on a current seaborn install raises `AttributeError: module 'seaborn' has no attribute 'distplot'`. It was split into two replacements:
- `sns.histplot()` — histogram, optionally with a smoothed density curve overlaid (`kde=True`)
- `sns.kdeplot()` — just the smooth density curve, no bars

`histplot(kde=True)` is the closest direct equivalent to what `distplot()` used to show.

In [ ]:
sns.histplot(df['Age'], kde=True)


## c. Boxplot

A boxplot summarizes a numeric column's spread visually: the box covers the 25th–75th percentile (IQR), the line inside is the median, the whiskers extend to the typical range, and individual dots beyond them are flagged as potential outliers.

*(Same positional-argument note as `countplot()` above — `x=` is the current, non-deprecated way to call it.)*

In [ ]:
sns.boxplot(df['Age'])

In [ ]:
sns.boxplot(x=df['Age'])


### Reading the Distribution Numerically

The plots above show the shape visually — these four numbers describe it precisely, and are worth computing alongside any distribution plot.

In [ ]:
df['Age'].min()

In [ ]:
df['Age'].max()

In [ ]:
df['Age'].mean()

In [ ]:
df['Age'].skew()

`.skew()` (already in the original notebook) measures **asymmetry** — 0 is perfectly symmetric, positive means a longer tail toward higher values. `.kurt()` measures **tailedness/peakedness** relative to a normal distribution — positive means heavier tails and a sharper peak (more extreme outliers than a normal distribution would predict).

In [ ]:
df['Age'].kurt()


## 3. Bivariate & Multivariate Analysis

Everything above looked at one column in isolation. The real insight in EDA usually comes from looking at **how two or more columns relate** — e.g. does survival differ by sex, or by class?

### a. Categorical vs. Numerical — `barplot()`

`sns.barplot()` plots the **mean** of a numeric column, grouped by a categorical one (with a confidence-interval error bar) — perfect for a question like "what's the survival rate in each class?".

In [ ]:
sns.barplot(x='Pclass', y='Survived', data=df)


### b. Two Categorical Columns Together — `hue`

Adding `hue=` to a `countplot()` splits each category into sub-bars by a second categorical column — e.g. survival count within each sex, side by side.

In [ ]:
sns.countplot(x='Survived', hue='Sex', data=df)


### c. Correlation Heatmap

Rather than reading `df.corr()`'s numbers as a table (as in the previous notebook), `sns.heatmap()` renders the same correlation matrix as color-coded cells — patterns jump out much faster visually, especially with more columns. `numeric_only=True` is required here for the same reason noted in the EDA notebook — `.corr()` errors on non-numeric columns by default on modern pandas.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')


### d. Two Numerical Columns — `scatterplot()`

For two continuous columns, a scatter plot shows the relationship point by point. Adding `hue='Survived'` colors each point by a third (categorical) column — a simple way to fold a third variable into a 2D plot.

In [ ]:
sns.scatterplot(x='Age', y='Fare', hue='Survived', data=df)


## Summary

| Plot | Use for | Method |
|---|---|---|
| Countplot | 1 categorical column | `sns.countplot(x=col, data=df)` |
| Pie chart | 1 categorical column (proportions) | `df[col].value_counts().plot(kind='pie')` |
| Histogram | 1 numerical column (shape) | `plt.hist(df[col], bins=...)` |
| Hist + density | 1 numerical column | `sns.histplot(df[col], kde=True)` *(replaces removed `distplot`)* |
| Boxplot | 1 numerical column (spread/outliers) | `sns.boxplot(x=df[col])` |
| Barplot | categorical vs. numerical mean | `sns.barplot(x=cat, y=num, data=df)` |
| Countplot + hue | 2 categorical columns | `sns.countplot(x=c1, hue=c2, data=df)` |
| Heatmap | numeric correlations | `sns.heatmap(df.corr(numeric_only=True), annot=True)` |
| Scatterplot | 2 numerical (+1 categorical via hue) | `sns.scatterplot(x=n1, y=n2, hue=cat, data=df)` |

### Deprecation notes for this seaborn version
- Pass plot variables as **keywords** (`x=`, `y=`, `hue=`), not positionally — `sns.countplot(df[col])` now errors.
- `sns.distplot()` is **removed** — use `sns.histplot(kde=True)` or `sns.kdeplot()` instead.